# FullStack AI · Projet d'infrastructure

**Objectif :** Infrastructure d'IA Stripe : assembler données, modèle et déploiement.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Projet · Infrastructure d'IA anti-fraude

L'équivalent du projet « Infrastructure d'IA Stripe » : tu construis le service qui décide, en quelques millisecondes, si un paiement en ligne est frauduleux. Du jeu de données à l'API, avec les vraies questions d'un ingénieur ML : quelles variables, quelle métrique, quel seuil, et comment servir le modèle.

Le jeu `paiements` contient 10 000 transactions, dont environ 1 % de fraudes :

In [ ]:
from paiements import charger
df = charger()
print(df.shape, f"{df['fraude'].mean():.2%} de fraudes")
df.head()

## Trois difficultés réelles

1. **Le déséquilibre** : avec 1 % de fraudes, un modèle qui dit toujours « pas de fraude » a 99 % d'exactitude, et ne sert à rien. On regarde l'**AUC** (capacité à classer les fraudes au-dessus des autres) et la **précision moyenne** (*average precision*, l'aire sous la courbe précision-rappel).
2. **Les variables** : le montant brut ou l'heure brute disent peu. « La carte et l'adresse IP viennent de pays différents », « le compte a moins de 30 jours », « il est entre minuit et 6 h » disent beaucoup. C'est l'**ingénierie des variables**.
3. **Le seuil** : laisser passer une fraude coûte son montant ; bloquer un client honnête coûte une vente et sa confiance (on l'estime ici à 5 €). Le bon seuil est celui qui **minimise le coût total**, pas 0,5.

La fonction `preparer(df)` qui construit ces variables est fournie dans les exercices 2 et 3.

**À faire :**

- [ ] Sauvegarder le modèle avec joblib et l'API dans un main.py FastAPI, avec la vraie bibliothèque
- [ ] Suivre 5 variantes du modèle avec MLflow (régression logistique, forêt, boosting, avec et sans les variables construites) et enregistrer le champion
- [ ] Écrire le Dockerfile, construire l'image et la lancer en local
- [ ] Déployer l'image sur une instance EC2 derrière Caddy, et mesurer la latence de /v1/score avec 100 requêtes
- [ ] Écrire un README avec l'architecture, la métrique choisie, le seuil et son coût

### Exercice 1 · Construire les variables

Écris `preparer(df)` qui renvoie un DataFrame (même index que `df`) avec 7 colonnes : `pays_diff` (1 si `pays_carte` ≠ `pays_ip`), `log_montant` (log(1 + montant / 100)), `nb_tx_24h` (inchangé), `compte_recent` (1 si `anciennete_compte_j` < 30), `appareil_nouveau` (inchangé), `nuit` (1 si `heure` < 6) et `cat_risque` (1 si la catégorie est « électronique » ou « jeux »). Les indicateurs sont des entiers 0 ou 1.

<details><summary>Indice</summary>

Construis un dictionnaire de colonnes, puis pd.DataFrame(..., index=df.index). (condition).astype(int) donne 0 ou 1.
</details>

In [ ]:
import numpy as np
import pandas as pd
from paiements import charger

df = charger()

def preparer(df):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
X = preparer(df)
attendues = ["pays_diff", "log_montant", "nb_tx_24h", "compte_recent", "appareil_nouveau", "nuit", "cat_risque"]
assert X is not None and sorted(X.columns) == sorted(attendues), f"Colonnes attendues : {attendues}."
assert len(X) == len(df) and (X.index == df.index).all(), "Garde le même index que df."
assert (X["pays_diff"] == (df["pays_carte"] != df["pays_ip"]).astype(int)).all(), "pays_diff vaut 1 quand les pays diffèrent."
assert np.allclose(X["log_montant"], np.log1p(df["montant"] / 100)), "log_montant = log(1 + montant / 100) : np.log1p."
assert (X["compte_recent"] == (df["anciennete_compte_j"] < 30)).all() and (X["nuit"] == (df["heure"] < 6)).all(), "compte_recent : moins de 30 jours ; nuit : avant 6 h."
assert X["cat_risque"].sum() == df["categorie"].isin(["électronique", "jeux"]).sum(), "cat_risque : électronique ou jeux (isin)."
assert set(X["pays_diff"].unique()) <= {0, 1} and X["nuit"].dtype.kind == "i", "Les indicateurs doivent être des entiers 0 ou 1 (.astype(int))."
print("Bravo, tout est correct.")

### Exercice 2 · Entraîner, évaluer, choisir le seuil

Le découpage est fourni. Entraîne `modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))` sur l'entraînement, calcule `proba_test` (probabilités de fraude sur le test), `auc` (`roc_auc_score`) et `ap` (`average_precision_score`). Puis écris `cout(seuil)` : somme des montants (`m_test`) des fraudes **non** détectées (proba < seuil) + 5 € par transaction honnête bloquée (proba ≥ seuil). Mets dans `seuil` la valeur de `np.linspace(0.01, 0.99, 99)` qui minimise ce coût.

<details><summary>Indice</summary>

Masques booléens : (y_test == 1).to_numpy() & (proba_test < seuil). np.argmin sur la liste des coûts donne l'indice du meilleur seuil.
</details>

In [ ]:
import numpy as np
import pandas as pd
from paiements import charger
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def preparer(df):
    return pd.DataFrame({
        "pays_diff": (df["pays_carte"] != df["pays_ip"]).astype(int),
        "log_montant": np.log1p(df["montant"] / 100),
        "nb_tx_24h": df["nb_tx_24h"],
        "compte_recent": (df["anciennete_compte_j"] < 30).astype(int),
        "appareil_nouveau": df["appareil_nouveau"],
        "nuit": (df["heure"] < 6).astype(int),
        "cat_risque": df["categorie"].isin(["électronique", "jeux"]).astype(int),
    }, index=df.index)

df = charger()
X, y = preparer(df), df["fraude"]
X_train, X_test, y_train, y_test, m_train, m_test = train_test_split(
    X, y, df["montant"], test_size=0.25, stratify=y, random_state=0)
from sklearn.metrics import roc_auc_score, average_precision_score

modele = None
proba_test = None
auc = ap = None

def cout(seuil):
    pass

seuil = None

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
assert proba_test is not None and len(proba_test) == len(y_test), "proba_test doit contenir une probabilité par transaction du test (predict_proba(...)[:, 1])."
assert abs(auc - roc_auc_score(y_test, proba_test)) < 1e-12 and auc > 0.9, f"Avec ces variables, l'AUC doit dépasser 0,9 ; tu as {auc}."
assert 0 < ap < 1, "ap doit être la précision moyenne (average_precision_score)."
pt = np.asarray(proba_test); yt = np.asarray(y_test); mt = np.asarray(m_test)
ref = lambda s: mt[(yt == 1) & (pt < s)].sum() + 5 * ((yt == 0) & (pt >= s)).sum()
assert abs(cout(0.3) - ref(0.3)) < 1e-6, f"cout(0.3) devrait valoir {ref(0.3):.2f} € ; tu as {cout(0.3)}."
grille = np.linspace(0.01, 0.99, 99)
assert seuil is not None and abs(ref(seuil) - min(ref(s) for s in grille)) < 1e-6, "seuil doit minimiser le coût sur np.linspace(0.01, 0.99, 99)."
assert seuil < 0.5, "Le coût d'une fraude ratée est bien plus grand que celui d'une fausse alerte : le seuil optimal est bas."
print("Bravo, tout est correct.")

### Exercice 3 · Servir le modèle

Le modèle entraîné et le `SEUIL` sont fournis. Crée un modèle pydantic `Transaction` avec les 8 colonnes brutes (`montant` float > 0, `heure` int de 0 à 23, `pays_carte` et `pays_ip` str, `nb_tx_24h` et `anciennete_compte_j` int ≥ 0, `appareil_nouveau` int de 0 à 1, `categorie` str). Crée `app` avec `GET /sante` et `POST /v1/score`, qui transforme la transaction en DataFrame d'une ligne, la passe dans `preparer` puis dans le modèle, et renvoie `{"score": <proba arrondie à 4 décimales>, "fraude": <score ≥ SEUIL>, "version": VERSION}`.

<details><summary>Indice</summary>

pd.DataFrame([tx.model_dump()]) crée un DataFrame d'une ligne avec les colonnes brutes ; preparer s'occupe du reste.
</details>

In [ ]:
import numpy as np
import pandas as pd
from paiements import charger
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def preparer(df):
    return pd.DataFrame({
        "pays_diff": (df["pays_carte"] != df["pays_ip"]).astype(int),
        "log_montant": np.log1p(df["montant"] / 100),
        "nb_tx_24h": df["nb_tx_24h"],
        "compte_recent": (df["anciennete_compte_j"] < 30).astype(int),
        "appareil_nouveau": df["appareil_nouveau"],
        "nuit": (df["heure"] < 6).astype(int),
        "cat_risque": df["categorie"].isin(["électronique", "jeux"]).astype(int),
    }, index=df.index)

df = charger()
X, y = preparer(df), df["fraude"]
X_train, X_test, y_train, y_test, m_train, m_test = train_test_split(
    X, y, df["montant"], test_size=0.25, stratify=y, random_state=0)
from minifast import FastAPI, TestClient
from pydantic import BaseModel, Field

modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_train, y_train)
SEUIL = 0.07
VERSION = "fraude-1.0.0"

app = FastAPI()

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import pandas as pd
c = TestClient(app)
assert c.get("/sante").status_code == 200, "Ajoute GET /sante."
sus = {"montant": 900, "heure": 3, "pays_carte": "FR", "pays_ip": "NG", "nb_tx_24h": 6, "anciennete_compte_j": 2, "appareil_nouveau": 1, "categorie": "électronique"}
sain = {"montant": 25, "heure": 14, "pays_carte": "FR", "pays_ip": "FR", "nb_tx_24h": 1, "anciennete_compte_j": 900, "appareil_nouveau": 0, "categorie": "alimentaire"}
r1, r2 = c.post("/v1/score", json=sus), c.post("/v1/score", json=sain)
assert r1.status_code == 200 and set(r1.json()) == {"score", "fraude", "version"}, f"La réponse doit contenir score, fraude et version ; tu as {r1.json()}."
attendu = float(modele.predict_proba(preparer(pd.DataFrame([sus])))[0, 1])
assert abs(r1.json()["score"] - round(attendu, 4)) < 1e-9, "Le score doit être la probabilité du modèle, après preparer, arrondie à 4 décimales."
assert r1.json()["fraude"] is True and r2.json()["fraude"] is False, "La transaction suspecte doit être bloquée et la transaction normale acceptée."
assert c.post("/v1/score", json={**sain, "montant": -10}).status_code == 422, "Un montant négatif doit être refusé (422)."
assert c.post("/v1/score", json={**sain, "heure": 25}).status_code == 422, "heure doit être entre 0 et 23."
print("Bravo, tout est correct.")